# Errors saved at a fixed coverage

Accept the 80% of calls the confidence score is most sure about. Among those accepted calls,
how many C0 errors does the final ensemble leave in, compared with the adjusted MSP?

* **risk@80%** = C0's error rate among the 80% most-confident calls.
* **fewer errors / 1,000 accepted** = (risk MSP − risk ensemble) × 1,000.
* **fewer errors / 1,000 calls** = the same × 0.8 (only 800 of 1,000 calls are accepted).

Pooled out-of-fold predictions; point estimates, no bootstrap interval. This is one point on
the risk-coverage curve -- E-AURC averages over all coverages, so the two need not agree.

In [1]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

sys.path.insert(0, os.path.abspath("code"))
from c2_msp import msp_score

RESULTS  = os.environ.get("THESIS_RESULTS", "/work3/s251710/thesis_results")
C2       = os.path.join(RESULTS, "C2_final_results")
DISEASES = ["effusion", "atelectasis", "cardiomegaly", "consolidation", "edema"]
K        = 10              # FINAL["k"] in C2_evaluation.ipynb
MSP_VARIANT = "threshold"  # the adjusted MSP
COVERAGE = 0.80

In [2]:
def c0_threshold(d):
    p = os.path.join(RESULTS, "C0_final", "multilabel_ignore", d, "threshold.txt")
    return float(open(p).read().strip())


def load(d):
    """Ensemble out-of-fold predictions joined to C0's probability. Returns the frame and the
    ensemble column -- the combiner with the best mean-fold AUROC, as best_ens() in
    C2_evaluation.ipynb picks it."""
    base = os.path.join(C2, d, "knn_correct_cf", f"cf_{K}")
    prob = pd.concat([pd.read_csv(os.path.join(base, "attributes", f"fold_{i}_predictions.csv"),
                                  usecols=["path", f"{d}_prob"]) for i in range(5)])
    df = pd.read_csv(os.path.join(base, "ensemble", "oof_predictions.csv")).merge(prob, on="path")
    fold_auc = lambda c: np.mean([roc_auc_score(g["correct"], g[c]) for _, g in df.groupby("fold")])
    col = max([c for c in df.columns if c.startswith("ens_")], key=fold_auc)
    df["msp"] = msp_score(df[f"{d}_prob"].values, c0_threshold(d), MSP_VARIANT)
    return df, col


def risk_at(correct, score, coverage):
    """C0 error rate among the `coverage` most-confident calls (ties broken stably)."""
    order = np.argsort(-np.asarray(score, float), kind="mergesort")
    k = int(round(coverage * len(order)))
    return 1 - np.asarray(correct)[order[:k]].mean()

In [3]:
rows = []
for d in DISEASES:
    df, col = load(d)
    r_msp = risk_at(df["correct"], df["msp"], COVERAGE)
    r_ens = risk_at(df["correct"], df[col], COVERAGE)
    rows.append({"disease": d, "combiner": col, "n": len(df),
                 "C0 error rate": 1 - df["correct"].mean(),
                 "risk MSP": r_msp, "risk ensemble": r_ens,
                 "fewer errors / 1,000 accepted": 1000 * (r_msp - r_ens),
                 "fewer errors / 1,000 calls": 1000 * COVERAGE * (r_msp - r_ens)})

table = pd.DataFrame(rows).set_index("disease")
table.loc["AVERAGE"] = table.select_dtypes("number").mean()
table.round(4)

,combiner,n,C0 error rate,risk MSP,risk ensemble,"fewer errors / 1,000 accepted","fewer errors / 1,000 calls"
disease,,,,,,,
effusion,ens_lr,90934.0,0.2056,0.1394,0.1373,2.1444,1.7155
atelectasis,ens_lr,80938.0,0.4054,0.3080,0.3097,-1.6680,-1.3344
cardiomegaly,ens_lr,92507.0,0.1897,0.0584,0.0581,0.2702,0.2162
consolidation,ens_rank,83688.0,0.3887,0.2602,0.2610,-0.7916,-0.6333
edema,ens_lr,89852.0,0.2517,0.1593,0.1571,2.1980,1.7584
AVERAGE,NaN,87583.8,0.2882,0.1851,0.1846,0.4306,0.3445


In [4]:
avg = table.loc["AVERAGE"]
print(f"At {COVERAGE:.0%} coverage the ensemble leaves {avg['fewer errors / 1,000 accepted']:.1f} "
      f"fewer errors per 1,000 accepted calls than the adjusted MSP "
      f"({avg['fewer errors / 1,000 calls']:.1f} per 1,000 calls overall), averaged over "
      f"{len(DISEASES)} tasks.")

At 80% coverage the ensemble leaves 0.4 fewer errors per 1,000 accepted calls than the adjusted MSP (0.3 per 1,000 calls overall), averaged over 5 tasks.


## Per side of C0's call

The same models, split by what C0 predicted: **positive** calls (TP vs FP) and **negative** calls
(TN vs FN). Within one side the adjusted MSP only orders calls by p, so its AUROC there is how
well C0's own confidence separates its right and wrong calls on that side; the ensemble's lead is
what it adds on top.

* **AUROC** = mean-fold AUROC on that side's calls only (folds with one outcome skipped).
* **share of deferrals** = of the (1 − COVERAGE) calls the global cut defers, the share on this
  side. The cut is the one above, over both sides at once.
* **errors deferred** = share of this side's C0 errors the cut defers.
* **risk kept** = C0's error rate among this side's accepted calls.

In [ ]:
SIDES = {1: "positive", 0: "negative"}


def side_auc(g, col):
    """Mean-fold AUROC on one side; folds where every call is right (or wrong) are skipped."""
    a = [roc_auc_score(f["correct"], f[col]) for _, f in g.groupby("fold") if f["correct"].nunique() == 2]
    return np.mean(a) if a else np.nan


def deferred(score, coverage):
    """True for the calls the global cut defers -- the complement of risk_at's accepted prefix."""
    order = np.argsort(-np.asarray(score, float), kind="mergesort")
    out = np.ones(len(order), bool)
    out[order[:int(round(coverage * len(order)))]] = False
    return out


side_rows = []
for d in DISEASES:
    df, col = load(d)
    df["pred"] = (df[f"{d}_prob"] >= c0_threshold(d)).astype(int)
    df["def_msp"], df["def_ens"] = deferred(df["msp"], COVERAGE), deferred(df[col], COVERAGE)
    for s, name in SIDES.items():
        g = df[df["pred"] == s]
        wrong = g["correct"] == 0
        r = {"disease": d, "side": name, "n": len(g), "share of calls": len(g) / len(df),
             "errors": int(wrong.sum()), "C0 error rate": wrong.mean(),
             "AUROC MSP": side_auc(g, "msp"), "AUROC ensemble": side_auc(g, col)}
        r["AUROC gain"] = r["AUROC ensemble"] - r["AUROC MSP"]
        for k, c in (("MSP", "def_msp"), ("ensemble", "def_ens")):
            r[f"share of deferrals {k}"] = g[c].sum() / df[c].sum()
            r[f"errors deferred {k}"]    = g.loc[wrong, c].mean()
            r[f"risk kept {k}"]          = 1 - g.loc[~g[c], "correct"].mean()
        side_rows.append(r)

side_table = pd.DataFrame(side_rows).set_index(["disease", "side"])
side_avg = side_table.groupby(level="side", sort=False).mean()
side_avg.index = pd.MultiIndex.from_product([["AVERAGE"], side_avg.index])
side_table = pd.concat([side_table, side_avg])
side_table.round(4)

In [ ]:
import importlib, thesis_style
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
importlib.reload(thesis_style)
from thesis_style import *
apply()

FIG_DIR = os.path.join("figures", "c2")
SIDE_TITLE = {"positive": "Positive calls (TP vs FP)", "negative": "Negative calls (TN vs FN)"}


def _signed(x, dec=3):
    return f"{x:+.{dec}f}".replace("-", "−")


def side_dumbbell(ax, side, order, ys):
    """One row per disease: adjusted MSP (grey) joined to the ensemble (purple, hollow if below
    MSP), the gain written right of the pair."""
    t = side_table.xs(side, level="side")
    for y, d in zip(ys, order):
        m, e = t.loc[d, "AUROC MSP"], t.loc[d, "AUROC ensemble"]
        ax.plot([m, e], [y, y], color=GREY, lw=LW["line"], zorder=1, solid_capstyle="butt")
        ax.scatter(m, y, s=16, color=GREY, lw=0, zorder=2)
        ax.scatter(e, y, s=18, facecolor=ENS if e >= m else "white", edgecolor=ENS, lw=0.9, zorder=3)
        ax.text(max(m, e) + 0.006, y, _signed(e - m), va="center", ha="left",
                fontsize=FS["tiny"], color=INK if e >= m else MUTED)
    ax.axhline((ys[-2] + ys[-1]) / 2, color=HAIR, lw=LW["hair"], zorder=0)
    ax.set_title(SIDE_TITLE[side])
    ax.set_xlabel("AUROC (correct vs wrong call)")
    ax.xaxis.grid(True)
    ax.set_axisbelow(True)


def side_figure(figsize=(118 * MM, 62 * MM)):
    # Ordered by C0's Youden threshold, low to high, as figure 2a of C2_evaluation.ipynb.
    order = sorted(DISEASES, key=c0_threshold) + ["AVERAGE"]
    ys    = list(range(len(order) - 1)) + [len(order) - 1 + 0.6]   # average set apart
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=True, sharey=True)
    for ax, side in zip(axes, SIDES.values()):
        side_dumbbell(ax, side, order, ys)
    lo = side_table[["AUROC MSP", "AUROC ensemble"]].min().min()
    hi = side_table[["AUROC MSP", "AUROC ensemble"]].max().max()
    axes[0].set_xlim(lo - 0.012, hi + 0.035)                        # room for the gain labels
    axes[0].set_yticks(ys, [d.capitalize() for d in order])
    axes[0].set_ylim(ys[-1] + 0.6, -0.6)
    axes[1].tick_params(axis="y", length=0)
    handles = [Line2D([], [], color=GREY, lw=0, marker="o", ms=4, label=f"MSP ({MSP_VARIANT})"),
               Line2D([], [], color=ENS, lw=0, marker="o", ms=4, label="Ensemble"),
               Line2D([], [], color=ENS, lw=0, marker="o", ms=4, mfc="white", mew=0.9,
                      label="Ensemble below MSP")]
    fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.07, 1, 1), w_pad=1.5)
    return fig


fig = side_figure()
os.makedirs(FIG_DIR, exist_ok=True)
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(FIG_DIR, f"knn_side_split_auroc.{ext}"))
plt.show()

## Per ground-truth class

The same models, split by the true label instead of C0's call: **truly positive** images (TP vs
FN: does the score flag the ones C0 missed?) and **truly negative** images (TN vs FP: does it flag
the false alarms?). `y` is not stored, but `pred = p >= t` and `correct = (pred == y)`, so
`y = pred` if correct, else `1 - pred`.

A diagnostic, not an operating point: at deployment `y` is unknown, so no cut can be set per
class. It says for which kind of patient the ensemble's ranking beats C0's own confidence.

* **AUROC** = mean-fold AUROC within the class (correct vs wrong), folds with one outcome skipped.
* **E-AURC** = mean-fold E-AURC within the class (lower = better); compared across classes only as
  the reduction in % of MSP's, because E-AURC grows with the error rate and the classes' differ.

In [ ]:
import c2_boot_extra as BX

CLASSES = {1: "positive", 0: "negative"}   # ground truth y


def fold_mean(g, col, metric):
    """Mean over folds of metric(correct, score); folds where every image is right (or wrong)
    are skipped."""
    v = [metric(f["correct"], f[col]) for _, f in g.groupby("fold") if f["correct"].nunique() == 2]
    return np.mean(v) if v else np.nan


gt_rows = []
for d in DISEASES:
    df, col = load(d)
    pred = (df[f"{d}_prob"] >= c0_threshold(d)).astype(int)
    df["y"] = np.where(df["correct"] == 1, pred, 1 - pred)
    for c, name in CLASSES.items():
        g = df[df["y"] == c]
        r = {"disease": d, "class": name, "n": len(g), "errors": int((g["correct"] == 0).sum()),
             "AUROC MSP":       fold_mean(g, "msp", roc_auc_score),
             "AUROC ensemble":  fold_mean(g, col,   roc_auc_score),
             "E-AURC MSP":      fold_mean(g, "msp", BX.e_aurc),
             "E-AURC ensemble": fold_mean(g, col,   BX.e_aurc)}
        r["C0 error rate"]      = r["errors"] / r["n"]
        r["AUROC gain"]         = r["AUROC ensemble"] - r["AUROC MSP"]
        r["E-AURC reduction %"] = 100 * (r["E-AURC MSP"] - r["E-AURC ensemble"]) / r["E-AURC MSP"]
        gt_rows.append(r)

gt_table = pd.DataFrame(gt_rows).set_index(["disease", "class"])
gt_avg = gt_table.groupby(level="class", sort=False).mean()
gt_avg.index = pd.MultiIndex.from_product([["AVERAGE"], gt_avg.index], names=gt_table.index.names)
gt_table = pd.concat([gt_table, gt_avg])
gt_table.round(4)

In [ ]:
import importlib, thesis_style
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.transforms import blended_transform_factory
importlib.reload(thesis_style)
from thesis_style import *
apply()

FIG_DIR  = os.path.join("figures", "c2")
GT_TITLE = {"positive": "Truly positive (TP vs FN)", "negative": "Truly negative (TN vs FP)"}


def _signed(x, dec=3):
    return f"{x:+.{dec}f}".replace("-", "−")


# metric -> (gt_table column, x label, point label, tick label, half-width of the x range, tick step)
GT_METRICS = {
    "auroc": ("AUROC gain", "AUROC gain over MSP", lambda g: _signed(g),
              lambda x: _signed(x), lambda m: np.ceil((m + 0.004) * 200) / 200, None),
    "eaurc": ("E-AURC reduction %", "E-AURC reduction vs MSP", lambda g: f"{_signed(g, 1)}%",
              lambda x: f"{_signed(x, 0)}%", lambda m: np.ceil((m + 2.5) / 2) * 2, 5),
}


def row_note(t, d):
    """The class's size and C0's error rate in it (AVERAGE: means over diseases)."""
    if d == "AVERAGE":
        ds = [x for x in t.index if x != "AVERAGE"]
        n, err = t.loc[ds, "n"].mean(), (t.loc[ds, "errors"] / t.loc[ds, "n"]).mean()
        return f"mean n={n:,.0f} · {100 * err:.1f}% wrong"
    return f"n={int(t.loc[d, 'n']):,} · {100 * t.loc[d, 'errors'] / t.loc[d, 'n']:.1f}% wrong"


def label_rows(ax, t, order, ys):
    """Disease name on the tick, this class's n and error rate under it in grey."""
    ax.set_yticks(ys, [d.capitalize() for d in order])
    ax.set_ylim(ys[-1] + 0.6, -0.6)
    left = blended_transform_factory(ax.transAxes, ax.transData)
    for y, d in zip(ys, order):
        ax.text(-0.035, y + 0.36, row_note(t, d), transform=left, ha="right", va="center",
                fontsize=FS["tiny"], color=MUTED)


def gt_rows_layout():
    # Ordered by C0's Youden threshold, low to high, as figure 2a of C2_evaluation.ipynb.
    order = sorted(DISEASES, key=c0_threshold) + ["AVERAGE"]
    ys    = list(range(len(order) - 1)) + [len(order) - 1 + 0.6]   # average set apart
    return order, ys


# ── The AUROCs themselves, before any subtraction ──
def gt_auroc_panel(ax, cls, order, ys):
    """One row per disease: MSP (grey) and the ensemble (purple, hollow if below MSP) on the full
    0-1 AUROC scale, both values written beside the pair. 0.5 = chance."""
    t = gt_table.xs(cls, level="class")
    ax.axvline(0.5, color=HAIR, ls="--", lw=LW["ref"], zorder=0)
    for y, d in zip(ys, order):
        m, e = t.loc[d, "AUROC MSP"], t.loc[d, "AUROC ensemble"]
        ax.plot([m, e], [y, y], color=GREY, lw=LW["line"], zorder=1, solid_capstyle="butt")
        ax.scatter(m, y, s=16, color=GREY, lw=0, zorder=2)
        ax.scatter(e, y, s=18, facecolor=ENS if e >= m else "white", edgecolor=ENS, lw=0.9, zorder=3)
        txt = f"MSP {m:.3f} · ens {e:.3f}"
        if max(m, e) < 0.5:     # near the floor: write to the right
            ax.text(max(m, e) + 0.04, y, txt, ha="left", va="center", fontsize=FS["tiny"], color=INK)
        else:                   # near the ceiling: write to the left
            ax.text(min(m, e) - 0.04, y, txt, ha="right", va="center", fontsize=FS["tiny"], color=INK)
    ax.axhline((ys[-2] + ys[-1]) / 2, color=HAIR, lw=LW["hair"], zorder=0)
    ax.set_xlim(-0.03, 1.03)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
    ax.set_title(GT_TITLE[cls])
    ax.set_xlabel("AUROC (correct vs wrong, within the class)")
    label_rows(ax, t, order, ys)


def gt_auroc_figure(figsize=(FULL, 72 * MM)):
    order, ys = gt_rows_layout()
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=True)   # own y labels: n differs by class
    for ax, cls in zip(axes, CLASSES.values()):
        gt_auroc_panel(ax, cls, order, ys)
    handles = [Line2D([], [], color=GREY, lw=0, marker="o", ms=4, label=f"MSP ({MSP_VARIANT})"),
               Line2D([], [], color=ENS, lw=0, marker="o", ms=4, label="Ensemble"),
               Line2D([], [], color=ENS, lw=0, marker="o", ms=4, mfc="white", mew=0.9,
                      label="Ensemble below MSP"),
               Line2D([], [], color=HAIR, ls="--", lw=LW["ref"], label="chance (0.5)")]
    fig.legend(handles=handles, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.07, 1, 1), w_pad=2.0)
    return fig


# ── Gains over MSP ──
def gt_panel(ax, cls, metric, order, ys, half):
    """One row per disease: the ensemble's gain over the adjusted MSP within one true class. The
    dashed line at 0 IS the MSP: right of it is a win, left a loss (hollow)."""
    col, xlabel, point_fmt, tick_fmt, _, step = GT_METRICS[metric]
    t = gt_table.xs(cls, level="class")
    ax.axvline(0, color=GREY, ls="--", lw=LW["ref"], zorder=1)
    for y, d in zip(ys, order):
        g = t.loc[d, col]
        ax.plot([0, g], [y, y], color=ENS, lw=LW["ref"], zorder=2, solid_capstyle="butt")
        ax.scatter(g, y, s=18, facecolor=ENS if g >= 0 else "white", edgecolor=ENS, lw=0.9, zorder=3)
        ax.text(g + 0.06 * half * (1 if g >= 0 else -1), y, point_fmt(g), ha="left" if g >= 0 else "right",
                va="center", fontsize=FS["tiny"], color=INK if g >= 0 else MUTED)
    ax.axhline((ys[-2] + ys[-1]) / 2, color=HAIR, lw=LW["hair"], zorder=0)
    ax.set_xlim(-half, half)
    if step:
        ax.xaxis.set_major_locator(plt.MultipleLocator(step))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: "0" if abs(x) < 1e-12 else tick_fmt(x)))
    ax.set_title(GT_TITLE[cls])
    ax.set_xlabel(xlabel)
    label_rows(ax, t, order, ys)


def gt_figure(metric, figsize=(FULL, 72 * MM)):
    order, ys = gt_rows_layout()
    col, *_, half_of, _ = GT_METRICS[metric]
    half  = half_of(gt_table[col].abs().max())                         # symmetric, both panels
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=True)   # own y labels: n differs by class
    for ax, cls in zip(axes, CLASSES.values()):
        gt_panel(ax, cls, metric, order, ys, half)
    handles = [Line2D([], [], color=GREY, ls="--", lw=LW["ref"], label=f"MSP ({MSP_VARIANT})"),
               Line2D([], [], color=ENS, lw=0, marker="o", ms=4, label="Ensemble"),
               Line2D([], [], color=ENS, lw=0, marker="o", ms=4, mfc="white", mew=0.9,
                      label="Ensemble worse than MSP")]
    fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.07, 1, 1), w_pad=2.0)
    return fig


os.makedirs(FIG_DIR, exist_ok=True)
figs = {"auroc_levels": gt_auroc_figure(), **{m: gt_figure(m) for m in GT_METRICS}}
for name, fig in figs.items():
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"knn_gt_split_{name}.{ext}"))
plt.show()

### Where the scores put each group

What the within-class AUROCs above are made of. One figure per true class, one row per disease,
MSP left and the ensemble right. Each panel shows where the score puts C0's **right** calls
(filled) and its **wrong** calls (outline) in that class.

The x-axis is the score's **confidence rank among all images of the fold** (0 = least confident,
1 = most). That puts MSP and the ensemble on one scale, and it is exactly what AUROC looks at:
AUROC = the chance that a random right call sits to the right of a random wrong one. Wrong calls
piled up on the right = AUROC near 0; piled up on the left = near 1. Each curve is normalised on
its own (the groups differ in size); the counts are next to the disease name.

In [ ]:
# (group, correct) per true class: the right calls first, drawn filled; the wrong ones in outline.
GT_GROUPS = {"positive": (("TP", 1), ("FN", 0)), "negative": (("TN", 1), ("FP", 0))}
SCORES    = {"MSP": (MUTED, f"MSP ({MSP_VARIANT})"), "ensemble": (ENS, "Ensemble")}  # darker grey than
BINS      = np.linspace(0, 1, 41)                                   # GREY, so outlines stay visible
Y_OF      = {name: y for y, name in CLASSES.items()}               # "positive" -> 1


def gt_frames():
    """Per disease: the true label y and both scores as their confidence rank within the fold
    (0 = least, 1 = most confident among ALL images of that fold, both classes)."""
    out = {}
    for d in DISEASES:
        df, col = load(d)
        pred = (df[f"{d}_prob"] >= c0_threshold(d)).astype(int)
        df["y"] = np.where(df["correct"] == 1, pred, 1 - pred)
        for s, c in (("MSP", "msp"), ("ensemble", col)):
            df[f"rank {s}"] = df.groupby("fold")[c].rank(pct=True)
        out[d] = df
    return out


def dist_panel(ax, g, cls, score):
    """Histogram of one score's rank for the right (filled) and wrong (outline) calls of a class."""
    color, _ = SCORES[score]
    for name, ok in GT_GROUPS[cls]:
        x = g.loc[g["correct"] == ok, f"rank {score}"]
        if ok:
            ax.hist(x, BINS, density=True, color=color, alpha=0.35, lw=0)
        else:
            ax.hist(x, BINS, density=True, histtype="step", color=color, lw=1.0)
    ax.set_xlim(0, 1)
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)


def gt_dist_figure(cls, frames, figsize=(FULL, 150 * MM)):
    order = sorted(DISEASES, key=c0_threshold)                 # as the other figures, no average
    fig, axes = plt.subplots(len(order), 2, figsize=figsize, sharex=True)
    t = gt_table.xs(cls, level="class")
    (right, _), (wrong, _) = GT_GROUPS[cls]
    for row, d in zip(axes, order):
        g = frames[d][frames[d]["y"] == Y_OF[cls]]
        for ax, score in zip(row, SCORES):
            dist_panel(ax, g, cls, score)
            auc = t.loc[d, f"AUROC {score}"]
            ax.text(0.01, 0.95, f"AUROC {auc:.3f}", transform=ax.transAxes, ha="left", va="top",
                    fontsize=FS["tiny"], color=INK)
        n_r, n_w = (g["correct"] == 1).sum(), (g["correct"] == 0).sum()
        row[0].set_ylabel(f"{d.capitalize()}\n{right} {n_r:,} · {wrong} {n_w:,}",
                          rotation=0, ha="right", va="center", fontsize=FS["small"])
    for ax, score in zip(axes[0], SCORES):
        ax.set_title(SCORES[score][1])
    for ax in axes[-1]:
        ax.set_xlabel("confidence rank among all images (0 = least, 1 = most confident)")
    handles = [Line2D([], [], color=MUTED, lw=0, marker="s", ms=6, alpha=0.35,
                      label=f"{right}: C0 right (filled)"),
               Line2D([], [], color=MUTED, lw=1.0, label=f"{wrong}: C0 wrong (outline)")]
    fig.legend(handles=handles, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(GT_TITLE[cls], x=0.02, ha="left")
    fig.tight_layout(rect=(0, 0.04, 1, 1), h_pad=0.6, w_pad=1.5)
    return fig


GT_FRAMES = gt_frames()
for cls in CLASSES.values():
    fig = gt_dist_figure(cls, GT_FRAMES)
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIG_DIR, f"knn_gt_split_dist_{cls}.{ext}"))
    plt.show()